# 01_dataset — LeRobotDataset を覗く

LeRobot v0.5.1 / `LeRobotDataset` (v3.0 形式) を読み込み、shape を確認し、画像を可視化する。

前提: `config.env` を `source` 済み（環境変数 `DATA_REPO`, `HF_HOME` が見える）。
計算ノードがオフラインの場合は事前DL（`env/predownload_hf.sh`）が済んでいること。

In [ ]:
import os

# config.env 由来の値を使う（直書きしない）。未設定なら fail-fast。
REPO_ID = os.environ.get("DATA_REPO")
assert REPO_ID and not REPO_ID.startswith("<TODO"), (
    "DATA_REPO が未設定です。先に `source config.env` してください。"
)
# オフライン環境ならコメントを外す（事前DL済みキャッシュのみ使う）。
# os.environ["HF_HUB_OFFLINE"] = "1"
print("DATA_REPO =", REPO_ID)
print("HF_HOME   =", os.environ.get("HF_HOME"))

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(REPO_ID)

# メタ情報: fps / エピソード数 / フレーム数
print("fps          =", dataset.meta.fps)
print("num_episodes =", dataset.num_episodes)
print("num_frames   =", dataset.num_frames)

In [ ]:
# 1 フレーム取り出して、どんなキーとテンソルがあるか確認する
sample = dataset[0]
for key, value in sample.items():
    shape = tuple(value.shape) if hasattr(value, "shape") else value
    print(f"{key:40s} {shape}")

In [ ]:
# action と state の shape を明示的に確認
print("action shape =", tuple(sample["action"].shape))
if "observation.state" in sample:
    print("state  shape =", tuple(sample["observation.state"].shape))

In [ ]:
# カメラ画像を 1 枚可視化する
import matplotlib.pyplot as plt

image_keys = [k for k in sample if k.startswith("observation.images")]
print("image keys:", image_keys)

if image_keys:
    img = sample[image_keys[0]]
    # LeRobot の画像テンソルは (C, H, W)。matplotlib 用に (H, W, C) へ。
    img_hwc = img.permute(1, 2, 0).cpu().numpy()
    plt.figure(figsize=(4, 4))
    plt.imshow(img_hwc)
    plt.title(image_keys[0])
    plt.axis("off")
    plt.show()

## delta_timestamps で時間窓を取る

`delta_timestamps`（現在フレーム t からの相対秒）を指定すると、同じキーに時間軸 `T` が増える。
ACT/Diffusion Policy が「複数ステップの観測・行動」を扱う直感をここで掴む。

In [ ]:
if image_keys:
    key = image_keys[0]
    delta_timestamps = {key: [-0.2, -0.1, 0.0]}  # 0.2s/0.1s 前 + 現在
    ds_t = LeRobotDataset(REPO_ID, delta_timestamps=delta_timestamps)
    s = ds_t[100]
    print(f"{key} shape with time window =", tuple(s[key].shape))  # 期待: (T, C, H, W), T=3